# Pharma Trial Intelligence with Snowflake

## 1. Business Problem
## 2. Data Sources
## 3. Snowflake Architecture
## 4. Raw Data Ingestion
## 5. Snowpark Transformations
## 6. Dynamic Table for KPI Layer
## 7. Semantic View
## 8. Business Questions
## 9. Interview Summary

# 1. Dependencies

In [2]:
import requests
import pandas as pd
import os

from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType, IntegerType, DateType

## 2. Snowflake connection

In [ ]:
connection_params = {
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "password": os.getenv("SNOWFLAKE_PASSWORD"),
    "role": "ACCOUNTADMIN",
    "warehouse": "COMPUTE_WH",
    "database": "PHARMA_ANALYTICS_DB",
    "schema": "PUBLIC"
}

session = Session.builder.configs(connection_params).create()
print("Connected to Snowflake")

## 3. Create databse/schema

In [ ]:
session.sql("CREATE DATABASE IF NOT EXISTS PHARMA_ANALYTICS_DB").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS PHARMA_ANALYTICS_DB.PUBLIC").collect()
session.sql("USE DATABASE PHARMA_ANALYTICS_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()

## 4. Extracting data

From ClinicalTrials.gov

In [ ]:
url = "https://clinicaltrials.gov/api/v2/studies"

params = {
    "query.cond": "cancer",
    "pageSize": 100
}

response = requests.get(url, params=params)
data = response.json()

studies = data["studies"]
len(studies)

## 5. Parsing JSON

In [ ]:
records = []

for study in studies:
    protocol = study.get("protocolSection", {})

    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    design = protocol.get("designModule", {})
    conditions = protocol.get("conditionsModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})

    nct_id = identification.get("nctId")
    title = identification.get("briefTitle")
    overall_status = status.get("overallStatus")
    start_date = status.get("startDateStruct", {}).get("date")
    completion_date = status.get("completionDateStruct", {}).get("date")

    phases = design.get("phases", [])
    phase = phases[0] if phases else None

    enrollment = design.get("enrollmentInfo", {}).get("count")

    condition_list = conditions.get("conditions", [])
    condition = condition_list[0] if condition_list else None

    lead_sponsor = sponsor.get("leadSponsor", {}).get("name")

    records.append({
        "nct_id": nct_id,
        "title": title,
        "overall_status": overall_status,
        "phase": phase,
        "enrollment": enrollment,
        "condition": condition,
        "lead_sponsor": lead_sponsor,
        "start_date": start_date,
        "completion_date": completion_date
    })

df_trials = pd.DataFrame(records)
df_trials.head()

## 6. Charging session in Snowflake

In [ ]:
snowpark_df = session.create_dataframe(df_trials)

snowpark_df.write.mode("overwrite").save_as_table("RAW_CLINICAL_TRIALS")

session.table("RAW_CLINICAL_TRIALS").show(10)

## 7. Transformation with Snowpark

In [ ]:
trials = session.table("RAW_CLINICAL_TRIALS")

clean_trials = (
    trials
    .select(
        F.col("NCT_ID"),
        F.col("TITLE"),
        F.upper(F.col("OVERALL_STATUS")).alias("STATUS"),
        F.upper(F.col("PHASE")).alias("PHASE"),
        F.col("ENROLLMENT").cast(IntegerType()).alias("ENROLLMENT"),
        F.upper(F.col("CONDITION")).alias("CONDITION"),
        F.upper(F.col("LEAD_SPONSOR")).alias("SPONSOR"),
        F.to_date(F.col("START_DATE")).alias("START_DATE"),
        F.to_date(F.col("COMPLETION_DATE")).alias("COMPLETION_DATE")
    )
    .filter(F.col("NCT_ID").is_not_null())
)

clean_trials.write.mode("overwrite").save_as_table("FACT_CLINICAL_TRIALS")

session.table("FACT_CLINICAL_TRIALS").show(10)

## 8. KPIs for Pharma

In [ ]:
kpis_by_phase = (
    session.table("FACT_CLINICAL_TRIALS")
    .group_by("PHASE", "STATUS")
    .agg(
        F.count("*").alias("TOTAL_TRIALS"),
        F.avg("ENROLLMENT").alias("AVG_ENROLLMENT")
    )
    .sort(F.col("TOTAL_TRIALS").desc())
)

kpis_by_phase.show()

## 9. Creating KPI table

In [ ]:
kpis_by_phase.write.mode("overwrite").save_as_table("KPI_TRIALS_BY_PHASE_STATUS")

## SQL for Dynamic Table

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE PHARMA_TRIAL_KPIS
TARGET_LAG = '1 hour'
WAREHOUSE = COMPUTE_WH
AS
SELECT
    PHASE,
    STATUS,
    CONDITION,
    SPONSOR,
    COUNT(*) AS TOTAL_TRIALS,
    AVG(ENROLLMENT) AS AVG_ENROLLMENT
FROM FACT_CLINICAL_TRIALS
GROUP BY PHASE, STATUS, CONDITION, SPONSOR;